In [1]:
#PREPARACIÓN DE ENTORNO (DEPENDENCIAS, IMPORTACIÓN DE DATASETS Y LOGINS)

In [2]:
#1. Instalar dependencias
!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets scikit-learn pandas
!pip install -q peft accelerate evaluate
!pip install -q kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [3]:
# 2.1. Rutas de los datasets en Kaggle
import kagglehub

# Dataset ISOT
path_isot = kagglehub.dataset_download("csmalarkodi/isot-fake-news-dataset")

# Dataset FakeNewsNet
path_fnn = kagglehub.dataset_download("mdepak/fakenewsnet")

# Dataset Welfake
path_welfake = kagglehub.dataset_download("studymart/welfake-dataset-for-fake-news")

print("ISOT path:", path_isot)
print("FakeNewsNet path:", path_fnn)
print("Welfake path:", path_welfake)

Using Colab cache for faster access to the 'isot-fake-news-dataset' dataset.
Using Colab cache for faster access to the 'fakenewsnet' dataset.
Using Colab cache for faster access to the 'welfake-dataset-for-fake-news' dataset.
ISOT path: /kaggle/input/isot-fake-news-dataset
FakeNewsNet path: /kaggle/input/fakenewsnet
Welfake path: /kaggle/input/welfake-dataset-for-fake-news


In [4]:
# 2.2. Descargar datasets de ISOT
import os

fake_csv_path = os.path.join(path_isot, "Fake.csv")
true_csv_path = os.path.join(path_isot, "True.csv")

print(fake_csv_path)
print(true_csv_path)

/kaggle/input/isot-fake-news-dataset/Fake.csv
/kaggle/input/isot-fake-news-dataset/True.csv


In [5]:
# 2.3. Descargar datasets de FakeNewsNet
buzz_fake_path = os.path.join(path_fnn, "BuzzFeed_fake_news_content.csv")
buzz_real_path = os.path.join(path_fnn, "BuzzFeed_real_news_content.csv")
politi_fake_path = os.path.join(path_fnn, "PolitiFact_fake_news_content.csv")
politi_real_path = os.path.join(path_fnn, "PolitiFact_real_news_content.csv")

print(buzz_fake_path)
print(buzz_real_path)
print(politi_fake_path)
print(politi_real_path)

/kaggle/input/fakenewsnet/BuzzFeed_fake_news_content.csv
/kaggle/input/fakenewsnet/BuzzFeed_real_news_content.csv
/kaggle/input/fakenewsnet/PolitiFact_fake_news_content.csv
/kaggle/input/fakenewsnet/PolitiFact_real_news_content.csv


In [6]:
# 2.4. Descargar datasets de Welfake
welfake_csv_path = os.path.join(path_welfake, "WELFake_Dataset.csv")

print(welfake_csv_path)

/kaggle/input/welfake-dataset-for-fake-news/WELFake_Dataset.csv


In [8]:
#3. Login en Hugging Face
from google.colab import userdata
from huggingface_hub import login

# Obtener secreto de Colab
token = userdata.get("HUGGINGFACE_TOKEN")

# Login en Hugging Face
login(token=token)

print("Token cargado correctamente:", token is not None)

Token cargado correctamente: True


In [9]:
#IMPLEMENTACIÓN DEL MODELO

In [10]:
#4. Imports
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
import evaluate

In [11]:
# 5.1. Función de carga de datasets genérica
def load_dataset(path, label):
    df = pd.read_csv(path)
    df = df[['title', 'text']].dropna()
    df['label'] = label
    return df

In [12]:
# 5.1.1 Función de carga de dataset específica para el dataset WELFake
# Dataset original:
#   0 = fake
#   1 = real
#
# Dataset entrenamiento:
#   0 = real
#   1 = fake

def load_dataset_welfake(path):
    df = pd.read_csv(path)

    # Nos quedamos con las columnas necesarias
    df = df[['title', 'text', 'label']].dropna()

    # Invertir labels:
    # original 1 -> nuevo 0
    # original 0 -> nuevo 1
    df['label'] = df['label'].apply(lambda x: 0 if x == 1 else 1)

    return df

In [13]:
# 5.2. Cargar datasets ISOT y recortar a 3000 filas
df_fake = load_dataset(fake_csv_path, 1).head(3000)
df_real = load_dataset(true_csv_path, 0).head(3000)

# 5.3. Cargar datasets FakeNewsNet completos
df_buzz_fake = load_dataset(buzz_fake_path, 1)
df_buzz_real = load_dataset(buzz_real_path, 0)
df_politi_fake = load_dataset(politi_fake_path, 1)
df_politi_real = load_dataset(politi_real_path, 0)

# 5.4. Cargar dataset Welfake y recortar a 3000 filas
df_welfake = load_dataset_welfake(welfake_csv_path).head(3000)

In [14]:
#Mostrar cabecera de cada datasets que conformará mi dataset para entrenamiento del modelo

In [15]:
df_fake.head()

,title,text,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1


In [16]:
df_real.head()

,title,text,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,0
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,0
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,0
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,0


In [17]:
df_buzz_fake.head()

,title,text,label
0,Proof The Mainstream Media Is Manipulating The...,I woke up this morning to find a variation of ...,1
1,Charity: Clinton Foundation Distributed “Water...,Former President Bill Clinton and his Clinton ...,1
2,A Hillary Clinton Administration May be Entire...,After collapsing just before trying to step in...,1
3,Trump’s Latest Campaign Promise May Be His Mos...,"Donald Trump is, well, deplorable. He’s sugges...",1
4,Website is Down For Maintenance,Website is Down For Maintenance,1


In [18]:
df_buzz_real.head()

,title,text,label
0,Another Terrorist Attack in NYC…Why Are we STI...,"On Saturday, September 17 at 8:30 pm EST, an e...",0
1,"Donald Trump: Drugs a 'Very, Very Big Factor' ...",Less than a day after protests over the police...,0
2,"Obama To UN: ‘Giving Up Liberty, Enhances Secu...","Obama To UN: ‘Giving Up Liberty, Enhances Secu...",0
3,Trump vs. Clinton: A Fundamental Clash over Ho...,Getty Images Wealth Of Nations Trump vs. Clint...,0
4,"President Obama Vetoes 9/11 Victims Bill, Sett...",President Obama today vetoed a bill that would...,0


In [19]:
df_politi_fake.head()

,title,text,label
0,Trump Just Insulted Millions Who Lost Everythi...,16.8k SHARES SHARE THIS STORY\n\nHillary Clint...,1
1,Famous dog killed in spot she waited a year fo...,Famous dog killed in spot she waited a year fo...,1
2,House oversight panel votes Clinton IT chief i...,Story highlights The House Oversight panel vot...,1
3,America Just Tragically Lost A Country Music I...,We are absolutely heartbroken to hear about th...,1
4,Monuments to the Battle for the New South,"Nine years ago, a driver lost control of his p...",1


In [20]:
df_politi_real.head()

,title,text,label
0,Trump Just Insulted Millions Who Lost Everythi...,16.8k SHARES SHARE THIS STORY\n\nHillary Clint...,0
1,Famous dog killed in spot she waited a year fo...,Famous dog killed in spot she waited a year fo...,0
2,House oversight panel votes Clinton IT chief i...,Story highlights The House Oversight panel vot...,0
3,America Just Tragically Lost A Country Music I...,We are absolutely heartbroken to hear about th...,0
4,Monuments to the Battle for the New South,"Nine years ago, a driver lost control of his p...",0


In [21]:
df_welfake.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,0
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",0
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,1
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",0
5,About Time! Christian Group Sues Amazon and SP...,All we can say on this one is it s about time ...,0


In [22]:
# 5.4. Unir todos los datasets en uno solo
df_total = pd.concat([
    df_fake,
    df_real,
    df_buzz_fake,
    df_buzz_real,
    df_politi_fake,
    df_politi_real,
    df_welfake
], ignore_index=True)

# 5.5. Crear columna "content" concatenando título + texto
df_total["content"] = df_total["title"] + " " + df_total["text"]

# 5.6. Comprobar distribución de etiquetas
print(df_total["label"].value_counts())

# 5.7. Mostrar las primeras filas de mi dataset para entrenar el modelo
df_total.head()

label
0    4797
1    4625
Name: count, dtype: int64


,title,text,label,content
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,1,Donald Trump Sends Out Embarrassing New Year’...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,1,Drunk Bragging Trump Staffer Started Russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",1,Sheriff David Clarke Becomes An Internet Joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",1,Trump Is So Obsessed He Even Has Obama’s Name...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,1,Pope Francis Just Called Out Donald Trump Dur...


In [23]:
# 6. Split train / validation / test (Dividir el dataset en entrenamiento, validación y prueba)
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

In [24]:
# Obtener un dataset para entrenamiento, otro para validación y otro para prueba
train_df, validation_df, test_df = random_split(df_total, 0.7, 0.1)

# Guardar los CSV
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

# Revisar el tamaño de cada dataset
print("Train:", len(train_df), "Validation:", len(validation_df), "Test:", len(test_df))

Train: 6595 Validation: 942 Test: 1885


In [25]:
#7. Creación de DataLoaders
from transformers import AutoTokenizer

model_name = "google/gemma-3-1b-pt"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Gemma no tiene pad token por defecto → lo igualamos al eos
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [26]:
#8. Crear dataset personalizado (CustomDataset)
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=512):
        self.data = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = self.data.iloc[index]["content"]
        label = self.data.iloc[index]["label"]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [27]:
#9. Crear datasets usando (CustomDataset)
train_dataset = CustomDataset(
    csv_file="train.csv",
    tokenizer=tokenizer,
    max_length=512
)

val_dataset = CustomDataset(
    csv_file="validation.csv",
    tokenizer=tokenizer,
    max_length=512
)

test_dataset = CustomDataset(
    csv_file="test.csv",
    tokenizer=tokenizer,
    max_length=512
)

In [28]:
#10. DataLoaders
from torch.utils.data import DataLoader

batch_size = 4  # Gemma 1B es más pesado que GPT2 pequeño

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size
)

In [29]:
#11. Verificación de dimensiones
for batch in train_loader:
    break

print("Input shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

Input shape: torch.Size([4, 512])
Attention mask shape: torch.Size([4, 512])
Labels shape: torch.Size([4])


In [30]:
#12. Verificación de formato
batch["input_ids"].dtype
batch["labels"].dtype

torch.int64

In [31]:
#13. Inicializamos el modelo base con pesos preentrenados
from transformers import AutoModelForCausalLM

model_name = "google/gemma-3-1b-pt"

base_model = AutoModelForCausalLM.from_pretrained(model_name)

base_model.eval()

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [32]:
#14. Comprobación generativa usando el modelo que acabamos de cargar
input_text = "Every effort moves you"
inputs = tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=20
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Every effort moves you towards the goal, but the moment you feel a bit weary is the moment where your journey becomes more


In [33]:
#15. Ejemplo de tipo clasificación de fake news usando prompting
text_2 = (
    "Is the following news fake? Answer with 'yes' or 'no': "
    "'Breaking: Scientists confirm the Earth is flat after new NASA study.'"
)

inputs = tokenizer(text_2, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=20
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Is the following news fake? Answer with 'yes' or 'no': 'Breaking: Scientists confirm the Earth is flat after new NASA study.'

<blockquote>'The study is the most detailed one ever made. They measured the Earth's curvature


El modelo base muestra capacidad de razonamiento semántico mediante prompting zero-shot, aunque no produce respuestas estrictamente estructuradas. Por ello, se procede a adaptar el modelo mediante fine-tuning supervisado con cabeza de clasificación.

In [34]:
#16. Cargar modelo base sin cabeza
from transformers import AutoModel

base_model = AutoModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [35]:
#17. Crear modelo con cabeza de clasificación
import torch.nn as nn
import torch

class GemmaForFakeNewsClassification(nn.Module):
    def __init__(self, base_model, hidden_size, num_classes=2):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Último token (como en el tutorial)
        last_hidden_state = outputs.last_hidden_state
        last_token = last_hidden_state[:, -1, :]

        logits = self.classifier(last_token)
        return logits

In [36]:
#18. Obtener hidden size automáticamente
hidden_size = base_model.config.hidden_size

model = GemmaForFakeNewsClassification(
    base_model=base_model,
    hidden_size=hidden_size,
    num_classes=2
)

model = model.to(base_model.dtype)

In [37]:
#20. Descongelar últimas N capas y la cabeza

N = 5  # Número de últimas capas a descongelar

# Congelar todo el backbone (el modelo base)
for param in model.base_model.parameters():
    param.requires_grad = False

# Descongelar las últimas N capas del transformer
for layer in model.base_model.layers[-N:]:  # Gemma3TextModel usa .layers
    for param in layer.parameters():
        param.requires_grad = True

# Descongelar la cabeza del clasificador
for param in model.classifier.parameters():
    param.requires_grad = True

In [38]:
#21. Verificación dimensional
batch = next(iter(train_loader))

input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

with torch.no_grad():
    outputs = model(input_ids, attention_mask)

print("Output shape:", outputs.shape)

Output shape: torch.Size([4, 2])


In [39]:
#22. Determinar si usar GPU o CPU (Device setup)
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device)
torch.manual_seed(123)

Device: cuda


In [40]:
#22. Obtener probabilidades y clase predicha
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    logits = model(input_ids, attention_mask)

probas = torch.softmax(logits, dim=-1)
predicted_label = torch.argmax(probas, dim=-1)

print("Predicted labels:", predicted_label)

Predicted labels: tensor([0, 0, 0, 0], device='cuda:0')


In [41]:
#23. Función de accuracy adaptada
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad():
            logits = model(input_ids, attention_mask)

        predicted_labels = torch.argmax(logits, dim=-1)

        num_examples += labels.size(0)
        correct_predictions += (predicted_labels == labels).sum().item()

    return correct_predictions / num_examples

In [42]:
#24. Accuracy inicial (antes de entrenar)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 40.00%
Validation accuracy: 47.50%
Test accuracy: 50.00%


In [43]:
#25. Loss batch adaptado
def calc_loss_batch(batch, model, device):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    logits = model(input_ids, attention_mask)
    loss = torch.nn.functional.cross_entropy(logits, labels)

    return loss

In [44]:
#26. Loss loader adaptado
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.

    if len(data_loader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break

        loss = calc_loss_batch(batch, model, device)
        total_loss += loss.item()

    return total_loss / num_batches

In [45]:
#27. Evaluación inicial de loss
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training loss: 0.916
Validation loss: 1.202
Test loss: 1.129


In [46]:
#FINETUNING EL MODELO

In [47]:
#28. Training loop adaptado
def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter):

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(batch, model, device)
            loss.backward()
            optimizer.step()

            examples_seen += batch["input_ids"].size(0)
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)

                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)

        print(f"Training accuracy: {train_accuracy*100:.2f}% | "
              f"Validation accuracy: {val_accuracy*100:.2f}%")

        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [48]:
#29. Función de evaluación
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [49]:
#30. Optimizer y entrenamiento
import time
import torch

start_time = time.time()
torch.manual_seed(123)

# Contar parámetros entrenables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable_params}/{total_params} ({trainable_params/total_params*100:.2f}%)")

# Crear optimizador solo sobre parámetros entrenables
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5,          # Bajo al descongelar varias capas
    weight_decay=0.1
)

num_epochs = 2

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs,
    eval_freq=50,
    eval_iter=5,
)

end_time = time.time()
print(f"Training completed in {(end_time - start_time)/60:.2f} minutes.")

Parámetros entrenables: 134212866/999888258 (13.42%)
Ep 1 (Step 000000): Train loss 0.836, Val loss 0.678
Ep 1 (Step 000050): Train loss 0.981, Val loss 0.431
Ep 1 (Step 000100): Train loss 0.759, Val loss 0.429
Ep 1 (Step 000150): Train loss 0.539, Val loss 0.669
Ep 1 (Step 000200): Train loss 0.824, Val loss 0.465
Ep 1 (Step 000250): Train loss 0.832, Val loss 0.507
Ep 1 (Step 000300): Train loss 0.682, Val loss 0.368
Ep 1 (Step 000350): Train loss 0.476, Val loss 0.535
Ep 1 (Step 000400): Train loss 0.775, Val loss 0.630
Ep 1 (Step 000450): Train loss 0.612, Val loss 0.481
Ep 1 (Step 000500): Train loss 0.553, Val loss 0.342
Ep 1 (Step 000550): Train loss 0.292, Val loss 0.437
Ep 1 (Step 000600): Train loss 0.257, Val loss 0.488
Ep 1 (Step 000650): Train loss 0.958, Val loss 0.347
Ep 1 (Step 000700): Train loss 0.302, Val loss 0.440
Ep 1 (Step 000750): Train loss 0.784, Val loss 0.447
Ep 1 (Step 000800): Train loss 0.652, Val loss 0.546
Ep 1 (Step 000850): Train loss 0.209, Val loss

In [50]:
#31. Accuracy final
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 91.40%
Validation accuracy: 82.59%
Test accuracy: 83.34%


In [51]:
# 32. Guardar SOLO últimas 5 capas + classifier
import torch
import os
import json

save_dir = "gemma_fake_news_adapter"
os.makedirs(save_dir, exist_ok=True)

N = 5  # últimas capas descongeladas

# Guardar solo capas necesarias
partial_state = {
    "last_layers": {
        k: v.cpu()
        for k, v in model.base_model.layers[-N:].state_dict().items()
    },
    "classifier": {
        k: v.cpu()
        for k, v in model.classifier.state_dict().items()
    },
    "config": {
        "model_name": "google/gemma-3-1b-pt",
        "num_unfrozen_layers": N,
        "hidden_size": model.classifier.in_features,
        "num_classes": model.classifier.out_features
    }
}

torch.save(partial_state, os.path.join(save_dir, "partial_weights.pt"))

# Guardar tokenizer
tokenizer.save_pretrained(save_dir)

print(f"Pesos parciales guardados en {save_dir}")

Pesos parciales guardados en gemma_fake_news_adapter
